In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import dataset
import dataset_misc1d
import space
from gp import creator as gp_creator
from gp import evaluator as gp_evaluator, selector as gp_selector
from gp import crossover as gp_crossover, mutator as gp_mutator
from gp import gp
from backprop.library import StaticLibrary
from symbols import syntax_tree
import randstate

np.seterr(all='ignore')

In [ ]:
SAMPLE_SIZE = 100
TRAIN_SIZE  = 0.7
NOISE       = 0.#05

POPSIZE          = 1000
MAX_STREE_DEPTH  = 8
MAX_STREE_LENGTH = 20
GENERATIONS      = 50
GROUP_SIZE       = 5  # tournament selector.
MUTATION_RATE    = 0.15
ELITISM          = 1

LIB_MAXDEPTH  = 3
RANDSTATE = 1234

In [ ]:
randstate.setstate(RANDSTATE)

S = dataset_misc1d.MagmanDatasetScaled()

S.sample(size=SAMPLE_SIZE, noise=NOISE, mesh=False)
#S.load('../data/magman.csv')

S.split(train_size=TRAIN_SIZE)
S.get_plotter().plot(width=8, height=6, plot_knowldege=False)

S_train = dataset.NumpyDataset(S)
S_test  = dataset.NumpyDataset(S, test=True)

In [ ]:
syntax_tree.SyntaxTreeInfo.set_problem(S_train)
lib = StaticLibrary(3, S_train, S_train.knowledge)

solutionCreator = gp_creator.PTC2RandomSolutionCreator(nvars=S.nvars)

multiMutator = gp_mutator.MultiMutator(
      gp_mutator.SubtreeReplacerMutator(MAX_STREE_DEPTH, MAX_STREE_LENGTH, solutionCreator),
      gp_mutator.FunctionSymbolMutator(),
      gp_mutator.NumericParameterMutator(all=True),
      gp_mutator.NumericParameterMutator(all=False)
      )

evaluator = gp_evaluator.MSEEvaluator(S_train)
selector  = gp_selector.TournamentSelector(GROUP_SIZE)


crossover = gp_crossover.ApproximatelyGeometricCrossover(MAX_STREE_DEPTH, MAX_STREE_LENGTH, S_train, lib)

settings = gp.GPSettings(
      POPSIZE, GENERATIONS, MAX_STREE_DEPTH, MAX_STREE_LENGTH, S_train, S_test,
      creator=solutionCreator,
      evaluator=evaluator,
      selector=selector,
      crossover=crossover,
      mutator=multiMutator,
      corrector=None,
      mutrate=MUTATION_RATE,
      elitism=ELITISM,
      knowledge=S.knowledge,
      track_fea_front=False)
symb_regressor = gp.GP(settings)

with tqdm(total=symb_regressor.ngen-1) as pbar:
      def on_newgen(genidx, status):
            pbar.update(1)
            pbar.set_description(status)
      best_stree, best_eval = symb_regressor.evolve(newgen_callback=on_newgen)

In [ ]:
test_data_evaluator = gp_evaluator.NMSEEvaluator(S_test)
best_stree.clear_output()
print("\n----- Best syntax tree -----")
print(best_stree)
print(f"Max depth: {best_stree.get_max_depth()}")
print(f"Length: {best_stree.get_nnodes()}")
print(f"Train {best_eval}")
print(f"Test NMSE: {test_data_evaluator.evaluate(best_stree).value}")

In [ ]:
best_stree.clear_output()
S.get_plotter().plot(width=8, height=6, plot_knowldege=False, model=best_stree, zoomout=1)

In [ ]:
symb_regressor.stats.plot()